# Dataset - DataFrame

### 1. Preparar el entorno e importar la clase de carga
El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar src/clases_po y se agrega src a sys.path. Así se puede importar HFDatasetLoader sin instalar el proyecto como paquete.

In [1]:
import sys
import pandas as pd
from pathlib import Path
import numpy as np

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

/Users/esteban.vasquez/Documents/personal/IUdigital/semestre 6/integrador3/Machine-Learning-/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2. Configurar el dataset y la salida
dataset_name identifica el repositorio en Hugging Face y split="train" indica la división que se descargará. El parámetro filename hace que el DataFrame también se guarde como CSV en src/carga_datos/datos/ cuando el notebook se ejecuta desde esa carpeta.

In [2]:
loader = HFDatasetLoader(
    dataset_name="sayanroy058/Business-Sales",
    split="train",
    filename="datos/Business-Sales.csv",
    
)

### 3. Descargar el dataset y crear el DataFrame
El método load() descarga el dataset, lo convierte a pandas.DataFrame y lo asigna a la variable df. La expresión df.head() muestra las primeras filas para comprobar que la carga fue exitosa.

In [3]:
df = loader.load()
df.head()


#Muestra porcentaje de nulos de todo el dataset

resumen_nulos = pd.DataFrame({
    'Total_Nulos': df.isnull().sum(),
    'Porcentaje_%': (df.isnull().mean() * 100).round(2)
})

# Mostrar solo las columnas que contienen al menos un nulo
print(resumen_nulos[resumen_nulos['Total_Nulos'] > 0])

              Total_Nulos  Porcentaje_%
make                10301          1.84
model               10399          1.86
trim                10651          1.91
body                13195          2.36
transmission        65352         11.69
vin                     4          0.00
condition           11820          2.12
odometer               94          0.02
color                 749          0.13
interior              749          0.13
mmr                    38          0.01
sellingprice           12          0.00
saledate               12          0.00


### 4. Limpieza de datos
Antes de limpiar se revisa la estructura original: 558.837 filas y 16 columnas. Cada fila es la venta de un vehículo usado en subasta, con sus características (año, marca, modelo, carrocería, millas recorridas, estado), el precio de referencia del mercado (`mmr`) y el precio final de venta (`sellingprice`).

In [4]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 558837 entries, 0 to 558836
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   year          558837 non-null  int64  
 1   make          548536 non-null  object 
 2   model         548438 non-null  object 
 3   trim          548186 non-null  object 
 4   body          545642 non-null  object 
 5   transmission  493485 non-null  object 
 6   vin           558833 non-null  object 
 7   state         558837 non-null  object 
 8   condition     547017 non-null  float64
 9   odometer      558743 non-null  float64
 10  color         558088 non-null  object 
 11  interior      558088 non-null  object 
 12  seller        558837 non-null  object 
 13  mmr           558799 non-null  float64
 14  sellingprice  558825 non-null  float64
 15  saledate      558825 non-null  object 
dtypes: float64(4), int64(1), object(11)
memory usage: 68.2+ MB


La limpieza se hace sobre el DataFrame completo (`df`) y no sobre la muestra, para que todas las filas queden con las mismas reglas.

- **Paso 1. Filas corridas de columna.** 26 filas de VW Jetta tienen un trim con coma ("SE PZEV w/Connectivity, Navitgation"). La coma corrió los valores una columna: `transmission` dice "sedan", el VIN quedó en `state` y el precio en `saledate`. Además les faltan `state` y `seller`, así que se eliminan.
- **Paso 2. Texto estandarizado.** El mismo valor aparece con mayúsculas distintas ("Ford" y "ford", "SUV" y "suv"). Se pasa todo a minúsculas, se quitan espacios sobrantes y se unifican las marcas escritas de varias formas ("chev truck", "vw", "mercedes-b", etc.).
- **Paso 3. Nulos ocultos.** En `color` e `interior` el valor "—" significa "sin dato". En `odometer` los valores 1 y 999.999 son marcadores de relleno. Todos se convierten a `NaN`.
- **Paso 4. Escala de `condition`.** Unas 70.000 filas usan la escala 1–5 y el resto la escala 10–50. El precio confirma que es la misma medida multiplicada por 10, por eso se multiplican por 10.
- **Paso 5. Fecha de venta.** `saledate` es texto ("Tue Dec 16 2014 12:30:00 GMT-0800 (PST)") y se convierte a `datetime`.
- **Paso 6. Duplicados.** Hay 87 ventas repetidas con el mismo VIN y la misma fecha, donde una de las dos filas tiene odómetro de relleno. Se conserva la fila con odómetro real. Los demás VIN repetidos son reventas del mismo auto en fechas distintas y se mantienen.
- **Paso 7. Tipos de dato.** Las columnas numéricas enteras pasan a `Int64` (entero que acepta nulos) y las columnas con pocos valores repetidos pasan a `category`.

In [5]:
df_limpio = df.copy()
cols_texto = ["make", "model", "trim", "body", "transmission", "state", "color", "interior", "seller"]

# PASO 1 - Eliminar filas corridas de columna
corridas = df_limpio["transmission"].str.lower().eq("sedan")
print("PASO 1 - Filas corridas eliminadas:", int(corridas.sum()))
df_limpio = df_limpio[~corridas].copy()

# PASO 2 - Minúsculas, espacios simples y marcas unificadas
print("\nPASO 2 - Valores únicos antes:",
      {c: df_limpio[c].nunique() for c in ["make", "model", "body", "transmission"]})

for c in cols_texto:
    df_limpio[c] = df_limpio[c].str.strip().str.lower().str.replace(r"\s+", " ", regex=True)

# Diccionario {como está escrito: como debe quedar}
df_limpio["make"] = df_limpio["make"].replace({
    "chev truck": "chevrolet", "ford tk": "ford", "ford truck": "ford",
    "dodge tk": "dodge", "gmc truck": "gmc", "hyundai tk": "hyundai",
    "mazda tk": "mazda", "landrover": "land rover", "mercedes": "mercedes-benz",
    "mercedes-b": "mercedes-benz", "vw": "volkswagen",
})
df_limpio["seller"] = df_limpio["seller"].str.replace("nissan-infiniti", "nissan infiniti", regex=False)

print("PASO 2 - Valores únicos después:",
      {c: df_limpio[c].nunique() for c in ["make", "model", "body", "transmission"]})

# PASO 3 - Nulos ocultos como texto o como número
print("\nPASO 3 - Valores '—' antes:",
      {c: int(df_limpio[c].eq("—").sum()) for c in ["color", "interior"]},
      "| odómetro 1 o 999999:", int(df_limpio["odometer"].isin([1, 999999]).sum()))

for c in ["color", "interior"]:
    df_limpio[c] = df_limpio[c].replace("—", np.nan)
df_limpio.loc[df_limpio["odometer"].isin([1, 999999]), "odometer"] = np.nan

print("PASO 3 - Nulos después:",
      {c: int(df_limpio[c].isna().sum()) for c in ["color", "interior", "odometer"]})

# PASO 4 - Una sola escala para condition (10 a 50)
print("\nPASO 4 - Filas con condition en escala 1-5:", int((df_limpio["condition"] <= 5).sum()))
df_limpio.loc[df_limpio["condition"] <= 5, "condition"] *= 10
print("PASO 4 - Rango de condition después:",
      df_limpio["condition"].min(), "a", df_limpio["condition"].max())

# PASO 5 - Fecha de venta como datetime
# Se toman los primeros 24 caracteres: "Tue Dec 16 2014 12:30:00"
df_limpio["saledate"] = pd.to_datetime(df_limpio["saledate"].str[:24],
                                       format="%a %b %d %Y %H:%M:%S", errors="coerce")
print("\nPASO 5 - Fechas de venta entre", df_limpio["saledate"].min(), "y", df_limpio["saledate"].max())

# PASO 6 - Duplicados por VIN y fecha (se conserva la fila con odómetro real)
filas_antes = len(df_limpio)
df_limpio = (df_limpio.sort_values("odometer")
             .drop_duplicates(["vin", "saledate"], keep="first")
             .sort_index())
print("\nPASO 6 - Duplicados eliminados:", filas_antes - len(df_limpio))

# PASO 7 - Tipos de dato
for c in ["make", "body", "transmission", "state", "color", "interior"]:
    df_limpio[c] = df_limpio[c].astype("category")
for c in ["condition", "odometer", "mmr", "sellingprice"]:
    df_limpio[c] = df_limpio[c].astype("Int64")

print("\nPASO 7 - Tipos:", df_limpio.dtypes.astype(str).to_dict())
print(f"\nFilas: {len(df):,} -> {len(df_limpio):,}")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1e6:.0f} MB -> "
      f"{df_limpio.memory_usage(deep=True).sum() / 1e6:.0f} MB")

df_limpio.head()

PASO 1 - Filas corridas eliminadas: 26



PASO 2 - Valores únicos antes: {'make': 96, 'model': 973, 'body': 86, 'transmission': 2}


PASO 2 - Valores únicos después: {'make': 55, 'model': 851, 'body': 45, 'transmission': 2}

PASO 3 - Valores '—' antes: {'color': 24685, 'interior': 17076} | odómetro 1 o 999999: 1387
PASO 3 - Nulos después: {'color': 25434, 'interior': 17825, 'odometer': 1481}

PASO 4 - Filas con condition en escala 1-5: 70101
PASO 4 - Rango de condition después: 10.0 a 50.0

PASO 5 - Fechas de venta entre 2014-01-01 09:15:00 y 2015-07-21 02:30:00



PASO 6 - Duplicados eliminados: 87



PASO 7 - Tipos: {'year': 'int64', 'make': 'category', 'model': 'object', 'trim': 'object', 'body': 'category', 'transmission': 'category', 'vin': 'object', 'state': 'category', 'condition': 'Int64', 'odometer': 'Int64', 'color': 'category', 'interior': 'category', 'seller': 'object', 'mmr': 'Int64', 'sellingprice': 'Int64', 'saledate': 'datetime64[ns]'}

Filas: 558,837 -> 558,724


Memoria: 389 MB -> 175 MB


,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,kia,sorento,lx,suv,automatic,5xyktca69fg566472,ca,50,16639,white,black,kia motors america inc,20500,21500,2014-12-16 12:30:00
1,2015,kia,sorento,lx,suv,automatic,5xyktca69fg561319,ca,50,9393,white,beige,kia motors america inc,20800,21500,2014-12-16 12:30:00
2,2014,bmw,3 series,328i sulev,sedan,automatic,wba3c1c51ek116351,ca,45,1331,gray,black,financial services remarketing (lease),31900,30000,2015-01-15 04:30:00
3,2015,volvo,s60,t5,sedan,automatic,yv1612tb4f1310987,ca,41,14282,white,black,volvo na rep/world omni,27500,27750,2015-01-29 04:30:00
4,2014,bmw,6 series gran coupe,650i,sedan,automatic,wba6b2c57ed129731,ca,43,2641,gray,black,financial services remarketing (lease),66000,67000,2014-12-18 12:30:00


### 5. Reducir el número de registros del DataFrame
Se toma una muestra aleatoria de 5.000 registros del DataFrame ya limpio (558.724 filas) para agilizar los reportes en Jupyter. Se usa `sample()` con `random_state=42` en lugar de las primeras 5.000 filas, porque el archivo está ordenado por lotes de subasta (las primeras filas son casi todas autos 2014–2015 de California) y esa porción sesgaría el análisis. La semilla fija hace que la muestra sea la misma en cada ejecución.

In [6]:
print("Número de registros en el dataset limpio:", df_limpio.shape)

df2 = df_limpio.sample(n=5000, random_state=42).reset_index(drop=True)
print("Nuevo dataset con 5000 registros:", df2.shape)

Número de registros en el dataset limpio: (558724, 16)
Nuevo dataset con 5000 registros: (5000, 16)


### 6. Revisar la estructura del DataFrame
Esta celda cuenta filas y columnas y muestra el tipo de cada columna junto con la memoria y los valores no nulos. Sirve para confirmar que la muestra conserva los tipos definidos en la limpieza.

In [7]:
print(f"Filas: {len(df2)}")
print(f"Columnas: {len(df2.columns)}")
df2.info()

Filas: 5000
Columnas: 16
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   year          5000 non-null   int64         
 1   make          4913 non-null   category      
 2   model         4912 non-null   object        
 3   trim          4910 non-null   object        
 4   body          4892 non-null   category      
 5   transmission  4414 non-null   category      
 6   vin           5000 non-null   object        
 7   state         5000 non-null   category      
 8   condition     4907 non-null   Int64         
 9   odometer      4991 non-null   Int64         
 10  color         4765 non-null   category      
 11  interior      4855 non-null   category      
 12  seller        5000 non-null   object        
 13  mmr           5000 non-null   Int64         
 14  sellingprice  5000 non-null   Int64         
 15  saledate     

In [8]:
df2.describe()

,year,condition,odometer,mmr,sellingprice,saledate
count,5000.000000,4907.0,4991.0,5000.0,5000.0,5000
mean,2010.078600,34.1129,68638.581647,13879.855,13701.2372,2015-03-05 03:45:50.772000
min,1985.000000,10.0,3.0,125.0,200.0,2014-01-07 00:00:00
25%,2008.000000,27.0,28602.5,7075.0,7000.0,2015-01-21 01:53:45
50%,2012.000000,36.0,52198.0,12400.0,12200.0,2015-02-13 01:00:00
75%,2013.000000,42.0,99359.0,18600.0,18400.0,2015-05-21 03:00:00
max,2015.000000,50.0,370466.0,99000.0,97000.0,2015-07-08 07:30:00
std,3.956053,9.421176,52890.335744,9642.906327,9650.4462,NaN


### 7. Ficha técnica y diagnóstico de la muestra
Resume el origen del dataset, los valores faltantes que quedan después de la limpieza y las estadísticas de las variables numéricas.

In [9]:
ficha_tecnica = pd.DataFrame({
    "Propiedad": [
        "Nombre del Dataset", "Repositorio / Identificador", "Enlace Directo",
        "Fecha de Consulta", "Licencia", "Contenido",
        "Periodo de ventas", "Muestra Procesada", "Uso de Memoria",
    ],
    "Detalle": [
        "Business Sales (ventas de vehículos usados en subasta)",
        "Hugging Face (sayanroy058/Business-Sales)",
        "https://huggingface.co/datasets/sayanroy058/Business-Sales",
        "4 de noviembre de 2026",
        "No declarada en el repositorio",
        "Características del vehículo, precio de referencia (mmr) y precio de venta",
        f"{df_limpio['saledate'].min():%Y-%m-%d} a {df_limpio['saledate'].max():%Y-%m-%d}",
        f"{df2.shape[0]} filas × {df2.shape[1]} columnas (de {len(df_limpio):,} filas limpias)",
        f"{df2.memory_usage(deep=True).sum() / 1024:.1f} KB",
    ],
})

numericas = ["year", "condition", "odometer", "mmr", "sellingprice"]


def auditar_df2(data):
    print("=== 1. FICHA TÉCNICA (Hugging Face) ===")
    print(ficha_tecnica.to_string(index=False))

    print("\n=== 2. DIAGNÓSTICO DE VALORES FALTANTES ===")
    faltantes = pd.DataFrame({
        "Completos": data.count(),
        "Faltantes": data.isna().sum(),
        "% Faltantes": (data.isna().mean() * 100).round(2),
    })
    print(faltantes[faltantes["Faltantes"] > 0])

    print("\n=== 3. DESCRIPCIÓN DE VARIABLES NUMÉRICAS ===")
    print(data[numericas].describe().T[["mean", "std", "min", "50%", "max"]])


auditar_df2(df2)

=== 1. FICHA TÉCNICA (Hugging Face) ===
                  Propiedad                                                                    Detalle
         Nombre del Dataset                     Business Sales (ventas de vehículos usados en subasta)
Repositorio / Identificador                                  Hugging Face (sayanroy058/Business-Sales)
             Enlace Directo                 https://huggingface.co/datasets/sayanroy058/Business-Sales
          Fecha de Consulta                                                     4 de noviembre de 2026
                   Licencia                                             No declarada en el repositorio
                  Contenido Características del vehículo, precio de referencia (mmr) y precio de venta
          Periodo de ventas                                                    2014-01-01 a 2015-07-21
          Muestra Procesada                        5000 filas × 16 columnas (de 558,724 filas limpias)
             Uso de Memoria      

# Reporte

### 8. Importar la clase de reportes
`Reportes` recibe un DataFrame en sus métodos y permite generar un resumen textual, imprimirlo en la consola o guardarlo en un archivo .txt.

In [10]:
from clases_po import Reportes

### 9. Generar y visualizar el reporte como texto
`generar_reporte(df)` devuelve un texto con el número de filas, columnas, duplicados, tipos de datos, valores nulos y estadísticas numéricas.

In [11]:
reportes = Reportes()
texto = reportes.generar_reporte(df2)

print(texto)

REPORTE DEL DATAFRAME
Filas: 5000
Columnas: 16
Duplicados: 0

Columnas y tipos:
- year: int64
- make: category
- model: object
- trim: object
- body: category
- transmission: category
- vin: object
- state: category
- condition: Int64
- odometer: Int64
- color: category
- interior: category
- seller: object
- mmr: Int64
- sellingprice: Int64
- saledate: datetime64[ns]

Valores nulos por columna:
- year: 0
- make: 87
- model: 88
- trim: 90
- body: 108
- transmission: 586
- vin: 0
- state: 0
- condition: 93
- odometer: 9
- color: 235
- interior: 145
- seller: 0
- mmr: 0
- sellingprice: 0
- saledate: 0

Estadísticas numéricas:
           year  condition   odometer        mmr  sellingprice
count  5000.000     4907.0     4991.0     5000.0        5000.0
mean   2010.079     34.113  68638.582  13879.855     13701.237
std       3.956      9.421  52890.336   9642.906      9650.446
min    1985.000       10.0        3.0      125.0         200.0
25%    2008.000       27.0    28602.5     7075.0     

### 10. Metadata descriptiva de `df2`
La tabla incluye tipo, longitud mínima, máxima y promedio en caracteres de los valores de cada columna, límites, conteos de nulos y no nulos, valores únicos, moda y frecuencia. En columnas numéricas también calcula rango, media, mediana, desviación estándar y cuartiles. Para texto, los límites son lexicográficos.

In [12]:
# reporte_descriptivo calcula min() y max(), que pandas no permite en columnas
# category sin orden. Para esta tabla se pasan temporalmente a texto (object).
cols_categoria = df2.select_dtypes("category").columns
metadata_df2 = reportes.reporte_descriptivo(df2.astype({c: "object" for c in cols_categoria}))
metadata_df2

,columna,tipo,longitud_minima_caracteres,longitud_maxima_caracteres,longitud_media_caracteres,valores_no_nulos,valores_nulos,porcentaje_nulos,valores_unicos,moda,frecuencia_moda,limite_inferior,limite_superior,rango,media,mediana,desviacion_estandar,cuartil_25,cuartil_75
0,year,int64,4,4,4.000000,5000,0,0.00,25,2012,960,1985,2015,30.0,2010.078600,2012.0,3.956053,2008.0,2013.0
1,make,object,3,13,5.986770,4913,87,1.74,43,ford,799,acura,volvo,NaN,NaN,NaN,NaN,NaN,NaN
2,model,object,1,24,6.806596,4912,88,1.76,459,altima,157,1 series,zephyr,NaN,NaN,NaN,NaN,NaN,NaN
3,trim,object,1,43,4.811202,4910,90,1.80,528,base,503,!,zx4 st,NaN,NaN,NaN,NaN,NaN,NaN
4,body,object,3,19,5.232011,4892,108,2.16,30,sedan,2121,access cab,wagon,NaN,NaN,NaN,NaN,NaN,NaN
5,transmission,object,6,9,8.893974,4414,586,11.72,2,automatic,4258,automatic,manual,NaN,NaN,NaN,NaN,NaN,NaN
6,vin,object,17,17,17.000000,5000,0,0.00,5000,19uua5647xa049403,1,19uua5647xa049403,zfbcfadh3ez024705,NaN,NaN,NaN,NaN,NaN,NaN
7,state,object,2,2,2.000000,5000,0,0.00,35,fl,734,ab,wi,NaN,NaN,NaN,NaN,NaN,NaN
8,condition,Int64,2,2,2.000000,4907,93,1.86,39,19,377,10,50,40.0,34.112900,36.0,9.421176,27.0,42.0
9,odometer,Int64,1,6,5.200761,4991,9,0.18,4910,9092,2,3,370466,370463.0,68638.581647,52198.0,52890.335744,28602.5,99359.0


### 11. Imprimir el reporte directamente
`imprimir_reporte(df)` genera el reporte y lo muestra de inmediato. Es útil para consultar el resumen durante una ejecución interactiva.

In [13]:
reportes.imprimir_reporte(df2)

REPORTE DEL DATAFRAME
Filas: 5000
Columnas: 16
Duplicados: 0

Columnas y tipos:
- year: int64
- make: category
- model: object
- trim: object
- body: category
- transmission: category
- vin: object
- state: category
- condition: Int64
- odometer: Int64
- color: category
- interior: category
- seller: object
- mmr: Int64
- sellingprice: Int64
- saledate: datetime64[ns]

Valores nulos por columna:
- year: 0
- make: 87
- model: 88
- trim: 90
- body: 108
- transmission: 586
- vin: 0
- state: 0
- condition: 93
- odometer: 9
- color: 235
- interior: 145
- seller: 0
- mmr: 0
- sellingprice: 0
- saledate: 0

Estadísticas numéricas:
           year  condition   odometer        mmr  sellingprice
count  5000.000     4907.0     4991.0     5000.0        5000.0
mean   2010.079     34.113  68638.582  13879.855     13701.237
std       3.956      9.421  52890.336   9642.906      9650.446
min    1985.000       10.0        3.0      125.0         200.0
25%    2008.000       27.0    28602.5     7075.0     

### 12. Guardar el reporte en un archivo de texto
`guardar_reporte(df, ruta)` crea los directorios necesarios y escribe el reporte en UTF-8. El archivo queda en `src/carga_datos/reportes/business-sales.txt` cuando el notebook se ejecuta desde `src/carga_datos`.

In [14]:
ruta = reportes.guardar_reporte(
    df2,
    "reportes/business-sales.txt",
)

print(f"Reporte guardado en: {ruta}")

Reporte guardado en: reportes/business-sales.txt


# Transformaciones

### 13. Transformación 1: Traducir y estandarizar nombres de columnas
Se renombran las 16 columnas del DataFrame de inglés a español con formato `snake_case` para mejorar la legibilidad del análisis.

In [15]:
columnas_espanol = {
    "year": "anio_modelo",
    "make": "marca",
    "model": "modelo",
    "trim": "version",
    "body": "carroceria",
    "transmission": "transmision",
    "vin": "vin",
    "state": "estado",
    "condition": "condicion",
    "odometer": "millas_recorridas",
    "color": "color",
    "interior": "color_interior",
    "seller": "vendedor",
    "mmr": "precio_referencia",
    "sellingprice": "precio_venta",
    "saledate": "fecha_venta",
}

df2 = df2.rename(columns=columnas_espanol)
df2.head()

,anio_modelo,marca,modelo,version,carroceria,transmision,vin,estado,condicion,millas_recorridas,color,color_interior,vendedor,precio_referencia,precio_venta,fecha_venta
0,2013,chrysler,town and country,touring-l,minivan,automatic,2c4rc1cgxdr723881,ny,38,28013,gray,black,chrysler capital,20800,19800,2015-06-17 03:15:00
1,2002,nissan,altima,2.5 sl,sedan,automatic,1n4al11d82c293575,ca,23,237972,gold,beige,aaero sweet company,275,1150,2014-12-17 11:30:00
2,2008,audi,q7,3.6 premium quattro,suv,automatic,wa1by74l28d034647,fl,28,74154,black,gray,elder ford of tampa,17650,16800,2014-12-18 09:45:00
3,2008,ford,f-150,stx,supercab,automatic,1ftrx14w68kd18180,ca,19,126674,blue,gray,skyhigh remarketing inc,8150,8600,2015-01-21 04:30:00
4,2003,mercedes-benz,c-class,c240,wagon,automatic,wdbrh61j53f361590,ca,36,53549,black,beige,hoehn motors inc,3250,7100,2015-01-21 04:30:00


### 14. Transformación 2: Edad del vehículo y uso por año
Se calcula la edad del vehículo al momento de la venta (año de la venta menos año del modelo) y las millas recorridas en promedio por año. Se usa la fecha de venta y no el año actual porque las ventas ocurrieron en 2014–2015. Un auto modelo 2015 vendido en 2014 daría edad −1, así que la edad mínima se deja en 0. Para autos de edad 0 se divide entre 1 año.

La columna `odometer` del dataset está en millas (las ventas son de Estados Unidos y Canadá), por eso se renombra como `millas_recorridas`.

In [16]:
df2["edad_vehiculo"] = (df2["fecha_venta"].dt.year - df2["anio_modelo"]).clip(lower=0).astype("Int64")
df2["millas_por_anio"] = (df2["millas_recorridas"] / df2["edad_vehiculo"].replace(0, 1)).round().astype("Int64")

df2[["marca", "modelo", "anio_modelo", "fecha_venta", "edad_vehiculo", "millas_recorridas", "millas_por_anio"]].head()

,marca,modelo,anio_modelo,fecha_venta,edad_vehiculo,millas_recorridas,millas_por_anio
0,chrysler,town and country,2013,2015-06-17 03:15:00,2,28013,14006
1,nissan,altima,2002,2014-12-17 11:30:00,12,237972,19831
2,audi,q7,2008,2014-12-18 09:45:00,6,74154,12359
3,ford,f-150,2008,2015-01-21 04:30:00,7,126674,18096
4,mercedes-benz,c-class,2003,2015-01-21 04:30:00,12,53549,4462


### 15. Transformación 3: Precio de venta frente al precio de referencia del mercado
`precio_referencia` (MMR, Manheim Market Report) es el valor estimado del vehículo en el mercado. Se calcula la diferencia en dólares, la proporción precio de venta / precio de referencia y una categoría que indica si el auto se vendió por debajo, dentro o por encima del valor de mercado (±10 %).

In [17]:
df2["dif_vs_referencia"] = df2["precio_venta"] - df2["precio_referencia"]
df2["ratio_precio_referencia"] = (df2["precio_venta"] / df2["precio_referencia"]).astype("Float64").round(3)

df2["posicion_mercado"] = pd.cut(
    df2["ratio_precio_referencia"],
    bins=[0, 0.9, 1.1, np.inf],
    labels=["debajo del mercado", "en el mercado", "encima del mercado"],
)

print(df2["posicion_mercado"].value_counts(dropna=False))
df2[["marca", "modelo", "precio_referencia", "precio_venta",
     "dif_vs_referencia", "ratio_precio_referencia", "posicion_mercado"]].head()

posicion_mercado
en el mercado         3211
debajo del mercado    1037
encima del mercado     752
Name: count, dtype: int64


,marca,modelo,precio_referencia,precio_venta,dif_vs_referencia,ratio_precio_referencia,posicion_mercado
0,chrysler,town and country,20800,19800,-1000,0.952,en el mercado
1,nissan,altima,275,1150,875,4.182,encima del mercado
2,audi,q7,17650,16800,-850,0.952,en el mercado
3,ford,f-150,8150,8600,450,1.055,en el mercado
4,mercedes-benz,c-class,3250,7100,3850,2.185,encima del mercado


### 16. Transformación 4: Variables de calendario de la venta
A partir de `fecha_venta` se extraen el mes y el día de la semana, para poder analizar si el precio cambia según el momento de la subasta.

In [18]:
dias = {"Monday": "lunes", "Tuesday": "martes", "Wednesday": "miércoles", "Thursday": "jueves",
        "Friday": "viernes", "Saturday": "sábado", "Sunday": "domingo"}

df2["mes_venta"] = df2["fecha_venta"].dt.month.astype("Int64")
df2["dia_semana_venta"] = df2["fecha_venta"].dt.day_name().map(dias)

df2[["fecha_venta", "mes_venta", "dia_semana_venta"]].head()

,fecha_venta,mes_venta,dia_semana_venta
0,2015-06-17 03:15:00,6,miércoles
1,2014-12-17 11:30:00,12,miércoles
2,2014-12-18 09:45:00,12,jueves
3,2015-01-21 04:30:00,1,miércoles
4,2015-01-21 04:30:00,1,miércoles


### 17. Transformación 5: Agrupamiento por marca
Se agrupan las ventas por marca para calcular el precio de venta promedio, el precio de referencia promedio, la condición y la edad promedio, el ratio precio/referencia promedio y la cantidad de ventas. Se muestran las 15 marcas con más ventas en la muestra.

In [19]:
df_resumen = (
    df2.groupby("marca", observed=True)
    .agg(
        precio_venta_promedio=("precio_venta", "mean"),
        precio_referencia_promedio=("precio_referencia", "mean"),
        condicion_promedio=("condicion", "mean"),
        edad_promedio=("edad_vehiculo", "mean"),
        ratio_promedio=("ratio_precio_referencia", "mean"),
        total_ventas=("vin", "count"),
    )
    .round(2)
    .sort_values("total_ventas", ascending=False)
    .reset_index()
)

df_resumen.head(15)

,marca,precio_venta_promedio,precio_referencia_promedio,condicion_promedio,edad_promedio,ratio_promedio,total_ventas
0,ford,14865.11,15147.34,35.78,4.01,0.99,799
1,chevrolet,12335.58,12548.1,32.74,4.71,0.97,513
2,nissan,11672.44,11782.73,34.13,4.19,1.0,508
3,toyota,12513.29,12622.89,33.54,4.72,1.0,380
4,dodge,11491.75,11639.25,32.99,4.38,0.98,300
5,honda,11185.12,11120.97,31.5,5.98,1.04,242
6,bmw,21260.35,21493.69,35.97,5.23,1.0,198
7,hyundai,11027.6,11304.23,35.27,3.4,0.97,183
8,chrysler,10371.34,10632.64,31.57,4.96,0.95,157
9,kia,11109.75,11410.23,37.42,3.1,0.98,154
